# Gradients and Backpropagation

We will trace a scalar network from a prediction error back to every parameter. Backpropagation applies the chain rule to calculate gradients: local sensitivities of a loss to parameter changes. We will compare a worked calculation with PyTorch autograd and an independent numerical approximation.

The model and targets are invented for arithmetic, not fitted to a dataset. Everything runs on CPU using PyTorch, without downloads or credentials. Run cells in order from a fresh kernel using the [README setup](README.md). The [notes](07_Gradients_and_Backpropagation_Notes.ipynb) provide optional background; all formulas and values needed for the computation are also included here.

## Start with the problem this lesson solves

A network made a wrong prediction, but many weights helped produce it. Which weights should change, in which direction, and by how much? Backpropagation answers the sensitivity part of that question by applying the chain rule backward through the forward calculation.

A gradient is a local rate of change of loss with respect to a parameter. It is not a new parameter value and not a moral assignment of blame. An optimizer uses these sensitivities to choose an update.

## Work forward, backward, and forward again

Use one training pair, input $x=2$ and target $y=1$. The hidden neuron has weight $w=0.5$ and bias $b=0.5$; the output has weight $v=2$ and bias $c=-0.5$.

**Forward calculation:**

$$z=wx+b=0.5(2)+0.5=1.5,$$
$$h=\max(0,z)=1.5,$$
$$\hat y=vh+c=2(1.5)-0.5=2.5,$$
$$L=\tfrac12(\hat y-y)^2=\tfrac12(2.5-1)^2=1.125.$$

The half is a convenient scaling choice: it cancels the two from differentiating a square. This is half squared error for one example, not the default scaling of PyTorch's MSE loss.

**Backward calculation, with each local multiplier visible:**

| Quantity | Reason | Loss derivative |
| --- | --- | --- |
| Prediction $\hat y$ | Derivative of half squared error is the error | $2.5-1=1.5$ |
| Output weight $v$ | Changing $v$ changes prediction by $h$ times that change | $1.5\times1.5=2.25$ |
| Output bias $c$ | It adds directly to prediction | $1.5\times1=1.5$ |
| Hidden output $h$ | Output multiplies $h$ by $v$ | $1.5\times2=3$ |
| Hidden score $z$ | ReLU's slope at positive 1.5 is one | $3\times1=3$ |
| Hidden weight $w$ | $z=wx+b$ has slope $x=2$ with respect to $w$ | $3\times2=6$ |
| Hidden bias $b$ | It adds directly to $z$ | $3\times1=3$ |

For example, $dL/dw=6$ says a sufficiently small increase $\Delta w$ changes loss by approximately $6\Delta w$, with other parameters fixed. It does not say that the weight equals six.

**Update all four parameters using their gradients at the old state.** With SGD rate $0.01$:

$$w'=0.5-0.01(6)=0.44,\quad b'=0.5-0.01(3)=0.47,$$
$$v'=2-0.01(2.25)=1.9775,\quad c'=-0.5-0.01(1.5)=-0.515.$$

**Repeat the forward pass:** $z'=0.44(2)+0.47=1.35$, so $h'=1.35$. The new prediction is $1.9775(1.35)-0.515=2.154625$. New loss is $\tfrac12(2.154625-1)^2\approx0.666579$, below $1.125$.

This is the complete learning mechanism on one example: predict, measure, differentiate, update, and predict again. The loss reduction is measured, not assumed; a larger step or another example could behave differently.

## Why passing gradients backward is useful

The output error can guide an early hidden weight even though that weight has no direct target. Backpropagation multiplies local sensitivities along paths and adds contributions when paths meet. Shared parameters therefore receive evidence from all their uses.

If this hidden score were negative, ReLU would give zero derivative along this path, preventing a local gradient update to its incoming weights from this example. Autograd follows the same mathematical rules automatically. Calling backward computes gradients; only a separate update changes parameters.

## Connect the explanation to the code

`requires_grad=True` asks autograd to track operations on a tensor. `loss.backward()` computes and accumulates gradients in `.grad`; it does not update weights. The manual dictionary contains the chain-rule values above. Finite differences perturb one parameter at a time to independently check the derivatives. Clearing accumulated gradients is necessary when the next batch should start a new gradient calculation.

The calculations above explain the mechanism before the full experiment. Read the following code cells in order; the printed values and assertions let you compare the implementation with its mathematical meaning.

In [1]:
import platform
import torch

torch.manual_seed(42)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
DTYPE = torch.float64
print("Python:", platform.python_version(), "| PyTorch:", torch.__version__)
print("CPU | double precision | fixed worked example")

Python: 3.9.6 | PyTorch: 2.2.2
CPU | double precision | fixed worked example


## Build the forward computation

The network is $z=wx+b$, $h=\operatorname{ReLU}(z)$, and $\hat y=vh+c$. Its scalar loss is $L=\tfrac12(\hat y-y)^2$. The factor of one half is explicit; PyTorch's default MSE does not include it.

`requires_grad=True` records how the loss depends on the parameters and the input. We call `retain_grad()` on intermediate tensors so their backward sensitivities will remain visible. This is only for inspection.

In [2]:
x = torch.tensor(2., dtype=DTYPE, requires_grad=True)
y = torch.tensor(1., dtype=DTYPE)
params = {name: torch.tensor(value, dtype=DTYPE, requires_grad=True)
          for name, value in {"w": 0.5, "b": 0.5, "v": 2., "c": -0.5}.items()}
z = params["w"] * x + params["b"]
h = torch.relu(z)
prediction = params["v"] * h + params["c"]
loss = 0.5 * (prediction - y).square()
for value in [z, h, prediction]:
    value.retain_grad()
for name, value in [("hidden score", z), ("hidden output", h), ("prediction", prediction), ("loss", loss)]:
    print(f"{name:14s}: {value.item():.3f}")
torch.testing.assert_close(loss, torch.tensor(1.125, dtype=DTYPE))

hidden score  : 1.500
hidden output : 1.500
prediction    : 2.500
loss          : 1.125


## Multiply local derivatives backward

The loss sensitivity to the prediction is the error. At the output, multiply that sensitivity by the hidden value to obtain the output-weight gradient, and by the output weight to carry it toward the hidden unit. ReLU contributes a factor of one here because its input is positive. The hidden weight then receives that signal multiplied by the input.

This calculation uses detached values deliberately so the manual arithmetic is separate from autograd's graph.

In [3]:
d_prediction = (prediction - y).detach()
d_h = d_prediction * params["v"].detach()
d_z = d_h * (z.detach() > 0).to(DTYPE)
manual = {
    "w": d_z * x.detach(),
    "b": d_z,
    "v": d_prediction * h.detach(),
    "c": d_prediction,
}
for name, derivative in manual.items():
    print(f"dL/d{name} = {derivative.item():.3f}")
print(f"dL/dx = {(d_z * params['w'].detach()).item():.3f}")

dL/dw = 6.000
dL/db = 3.000
dL/dv = 2.250
dL/dc = 1.500
dL/dx = 1.500


## Let autograd traverse the same graph

`backward()` accumulates derivatives into `.grad`; it does not change the parameters. We compare every parameter gradient with the manual result and also inspect the retained intermediate gradients. A snapshot establishes that the stored parameter values stayed fixed.

In [4]:
before = {name: value.detach().clone() for name, value in params.items()}
loss.backward()
for name, value in params.items():
    torch.testing.assert_close(value.grad, manual[name])
    torch.testing.assert_close(value, before[name])
    print(f"{name}: value={value.item():.3f}, gradient={value.grad.item():.3f}")
for name, value in [("prediction", prediction), ("h", h), ("z", z), ("x", x)]:
    print(f"Loss sensitivity to {name}: {value.grad.item():.3f}")
torch.testing.assert_close(x.grad, torch.tensor(1.5, dtype=DTYPE))
print("Manual and autograd gradients agree; parameters are unchanged.")

w: value=0.500, gradient=6.000
b: value=0.500, gradient=3.000
v: value=2.000, gradient=2.250
c: value=-0.500, gradient=1.500
Loss sensitivity to prediction: 1.500
Loss sensitivity to h: 3.000
Loss sensitivity to z: 3.000
Loss sensitivity to x: 1.500
Manual and autograd gradients agree; parameters are unchanged.


## Compare with small numerical perturbations

The function below evaluates the same loss from a parameter vector ordered as weight, bias, output weight, output bias. Central differences perturb one parameter at a time while holding the others fixed. Double precision and a small perturbation keep rounding error modest; all perturbed hidden scores remain away from the ReLU corner.

In [5]:
def scalar_loss(values):
    w, b, v, c = values
    estimate = v * torch.relu(w * 2. + b) + c
    return 0.5 * (estimate - 1.).square()

theta = torch.stack([before[name] for name in ["w", "b", "v", "c"]])
epsilon = 1e-6
numerical = []
for index in range(theta.numel()):
    offset = torch.zeros_like(theta)
    offset[index] = epsilon
    numerical.append((scalar_loss(theta + offset) - scalar_loss(theta - offset)) / (2 * epsilon))
numerical = torch.stack(numerical)
analytical = torch.stack([manual[name] for name in ["w", "b", "v", "c"]])
torch.testing.assert_close(numerical, analytical, rtol=1e-6, atol=1e-8)
print("Autograd/manual:", analytical.tolist())
print("Central differences:", numerical.tolist())
print("Largest absolute difference:", (numerical - analytical).abs().max().item())

Autograd/manual: [6.0, 3.0, 2.25, 1.5]
Central differences: [5.999999999839467, 3.000000000086267, 2.2499999999814335, 1.500000000209667]
Largest absolute difference: 2.0966695046809036e-10


## Apply one explicitly separate parameter change

Subtract a small multiple of the gradient from a detached copy of the parameters. This is a single illustrative gradient step, not a training loop. Its loss decreases on this example, although larger steps are not guaranteed to do so. The original parameters remain available for the following demonstrations.

In [6]:
step_size = 0.01
updated = theta - step_size * analytical
updated_loss = scalar_loss(updated)
assert updated_loss < scalar_loss(theta)
print("Updated parameters:", updated.tolist())
print(f"Loss before: {scalar_loss(theta).item():.6f}")
print(f"Loss after:  {updated_loss.item():.6f}")

Updated parameters: [0.44, 0.47, 1.9775, -0.515]
Loss before: 1.125000
Loss after:  0.666579


## Follow an inactive ReLU path

With input minus two, the original hidden score is negative and ReLU returns zero. Its derivative blocks the gradient toward the hidden parameters. The output weight multiplies zero, but the output bias still affects the prediction. Fresh leaf tensors keep this demonstration independent of earlier gradients.

In [7]:
inactive = theta.clone().requires_grad_()
w, b, v, c = inactive.unbind()
negative_hidden_score = w * (-2.) + b
negative_prediction = v * torch.relu(negative_hidden_score) + c
negative_loss = 0.5 * (negative_prediction - 1.).square()
negative_loss.backward()
expected_inactive = torch.tensor([0., 0., 0., -1.5], dtype=DTYPE)
torch.testing.assert_close(inactive.grad, expected_inactive)
print("Hidden score:", negative_hidden_score.item(), "| loss:", negative_loss.item())
print("Gradients [w, b, v, c]:", inactive.grad.tolist())

Hidden score: -0.5 | loss: 1.125
Gradients [w, b, v, c]: [-0.0, 0.0, -0.0, -1.5]


## Shared parameters collect contributions from a batch

For two examples, the objective is the mean of their half-squared errors. The gradient must therefore be the mean of their individual parameter gradients. `torch.autograd.grad` returns the requested derivatives directly, making the comparison possible without using an accumulating `.grad` buffer.

In [8]:
batch_x = torch.tensor([2., 1.], dtype=DTYPE)
batch_y = torch.tensor([1., 0.], dtype=DTYPE)
def batch_loss(values, inputs, targets):
    w, b, v, c = values
    guesses = v * torch.relu(w * inputs + b) + c
    return (0.5 * (guesses - targets).square()).mean()

batch_params = theta.clone().requires_grad_()
batch_gradient, = torch.autograd.grad(batch_loss(batch_params, batch_x, batch_y), batch_params)
individual_gradients = []
for input_value, target_value in zip(batch_x, batch_y):
    row_params = theta.clone().requires_grad_()
    gradient, = torch.autograd.grad(batch_loss(row_params, input_value, target_value), row_params)
    individual_gradients.append(gradient)
mean_gradient = torch.stack(individual_gradients).mean(dim=0)
torch.testing.assert_close(batch_gradient, mean_gradient)
torch.testing.assert_close(batch_gradient, torch.tensor([4.5, 3., 1.875, 1.5], dtype=DTYPE))
print("Individual gradients:\n", torch.stack(individual_gradients))
print("Mean-loss gradient:", batch_gradient.tolist())

Individual gradients:
 tensor([[6.0000, 3.0000, 2.2500, 1.5000],
        [3.0000, 3.0000, 1.5000, 1.5000]], dtype=torch.float64)
Mean-loss gradient: [4.5, 3.0, 1.875, 1.5]


## Make gradient accumulation visible

Two new forward/backward computations on the same leaf tensor add their contributions to `.grad`. Clearing the buffer separates independent calculations. Each call below constructs a fresh graph, so there is no need to retain a used graph in memory. No parameter updates occur in this demonstration.

In [9]:
accumulating = theta.clone().requires_grad_()
scalar_loss(accumulating).backward()
first_gradient = accumulating.grad.clone()
scalar_loss(accumulating).backward()
torch.testing.assert_close(accumulating.grad, 2 * first_gradient)
print("After one backward:", first_gradient.tolist())
print("After two backward calls:", accumulating.grad.tolist())
accumulating.grad = None
scalar_loss(accumulating).backward()
torch.testing.assert_close(accumulating.grad, first_gradient)
print("After clearing and recomputing:", accumulating.grad.tolist())

After one backward: [6.0, 3.0, 2.25, 1.5]
After two backward calls: [12.0, 6.0, 4.5, 3.0]
After clearing and recomputing: [6.0, 3.0, 2.25, 1.5]


## What the calculations show

Backpropagation reproduced the worked chain-rule derivatives, and numerical perturbations independently agreed. The gradients appeared without changing any parameter. An explicit small step then reduced this example's loss. The inactive-path and batch examples showed why derivatives can become zero on one path or collect contributions from several paths.

These computations demonstrate gradient mechanics, not model quality. A zero derivative is not proof of a solved task, and a successful backward call does not establish that the chosen objective, data, or computation is appropriate.

In [ ]:
torch.manual_seed(5)
A0 = torch.tensor([[1., 2.], [-1., 1.]])
W1 = torch.tensor([[.2, -.3], [.4, .1]], requires_grad=True); b1 = torch.tensor([.1, -.2], requires_grad=True)
W2 = torch.tensor([[.5, -.7]], requires_grad=True); b2 = torch.tensor([.3], requires_grad=True)
Z1 = A0 @ W1.T + b1; A1 = torch.tanh(Z1); Z2 = A1 @ W2.T + b2
loss = .5 * Z2.square().mean(); loss.backward()
with torch.no_grad():
    delta2 = Z2 / A0.shape[0]
    expected_W2 = delta2.T @ A1
    expected_b2 = delta2.sum(0)
    delta1 = (delta2 @ W2) * (1 - A1.square())
    expected_W1 = delta1.T @ A0
    expected_b1 = delta1.sum(0)
torch.testing.assert_close(W2.grad, expected_W2); torch.testing.assert_close(b2.grad, expected_b2)
torch.testing.assert_close(W1.grad, expected_W1); torch.testing.assert_close(b1.grad, expected_b1)
print('matrix gradients verified:', W1.grad.shape, b1.grad.shape, W2.grad.shape, b2.grad.shape)